# Model Monitoring

## Importing Libraries and Setting up Paths

In [ ]:
%%time

from datetime import datetime, timedelta, timezone
import json
import os
import re
import boto3
from time import sleep
from threading import Thread

import pandas as pd

from sagemaker import get_execution_role, session, Session, image_uris
from sagemaker.s3 import S3Downloader, S3Uploader
from sagemaker.processing import ProcessingJob
from sagemaker.serializers import CSVSerializer

from sagemaker.model import Model
from sagemaker.model_monitor import DataCaptureConfig

session = Session()

In [ ]:
from sagemaker.model_monitor import ModelQualityMonitor
from sagemaker.model_monitor import EndpointInput
from sagemaker.model_monitor.dataset_format import DatasetFormat

In [ ]:
# Get Execution role
role = get_execution_role()
print("RoleArn:", role)

region = session.boto_region_name
print("Region:", region)

In [ ]:
bucket = session.default_bucket()
print("Churn Bucket:", bucket)
prefix = "sagemaker/InternetChurn-ModelQualityMonitor"

##S3 prefixes
data_capture_prefix = f"{prefix}/datacapture"
s3_capture_upload_path = f"s3://{bucket}/{data_capture_prefix}"

ground_truth_upload_path = (
    f"s3://{bucket}/{prefix}/ground_truth_data/{datetime.now():%Y-%m-%d-%H-%M-%S}"
)

reports_prefix = f"{prefix}/reports"
s3_report_path = f"s3://{bucket}/{reports_prefix}"

##Get the model monitor image
monitor_image_uri = image_uris.retrieve(framework="model-monitor", region=region)

print("Image URI:", monitor_image_uri)
print(f"Capture path: {s3_capture_upload_path}")
print(f"Ground truth path: {ground_truth_upload_path}")
print(f"Report path: {s3_report_path}")

In [ ]:
model_url = 's3://sagemaker-us-east-1-035349649198/xgboost-project/xgb_output/xgb-2024-02-23-01-25-43/xgb-2024-02-23-01-25-43/output/model.tar.gz'
model_url

## Deploying Trained XGBoost Model

In [ ]:
model_name = f"xgb-churn-pred-model-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"

# Model version 1.7-1
image_uri = image_uris.retrieve(framework="xgboost", version="1.7-1", region=region)

model = Model(image_uri=image_uri, model_data=model_url, role=role, sagemaker_session=session)

In [ ]:
# Creating Endpoint
endpoint_name = f"xgb-churn-model-quality-monitor-{datetime.utcnow():%Y-%m-%d-%H%M}"
print("EndpointName =", endpoint_name)

# Enable Data Capture
data_capture_config = DataCaptureConfig(
    enable_capture=True, sampling_percentage=100, destination_s3_uri=s3_capture_upload_path
)

# Deploy pretrained model
model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.xlarge",
    endpoint_name=endpoint_name,
    data_capture_config=data_capture_config,
)

In [ ]:
from sagemaker.predictor import Predictor
# Creating predictor object from endpoint just invoked
predictor = Predictor(endpoint_name=endpoint_name, sagemaker_session=session, serializer=CSVSerializer())

## Setting up Baseline Dataset

In [ ]:
# Choosing a value for cutoff that is considered a customer will churn
churn_cutoff = 0.7
# Creating empty csv to fill with probability and predictions
validate_dataset = "validations_predictions.csv"

In [ ]:
# Prof instructed needed 200 samples to compute sd
limit = 200 
i = 0
# Opening validation data
with open(f"test_data/{validate_dataset}", "w") as baseline_file:
    # Writing probability and prediciton
    baseline_file.write("probability,prediction,label\n") 
    with open("test_data/val (5).csv", "r") as f:
        for row in f:
            (label, input_cols) = row.split(",", 1)
            #print(f"Sending data for prediction: {input_cols}")
            #output = predictor.predict(input_cols)
            #print(output)
            # Getting probability with predictor object
            probability = float(predictor.predict(input_cols))
            # Getting prediction with cutoff value
            prediction = "1" if probability > churn_cutoff else "0"
            baseline_file.write(f"{probability},{prediction},{label}\n")
            i += 1
            if i > limit:
                break
            print(".", end="", flush=True)
            sleep(0.5)
print()
print("Done!")

In [ ]:
# Ensuring file has populated
!head test_data/validations_predictions.csv

In [ ]:
# Uploading predictions using validation set to get model quality
# baseline stats
baseline_prefix = prefix + "/baselining"
baseline_data_prefix = baseline_prefix + "/data"
baseline_results_prefix = baseline_prefix + "/results"
# Setting up path
baseline_data_uri = f"s3://{bucket}/{baseline_data_prefix}"
baseline_results_uri = f"s3://{bucket}/{baseline_results_prefix}"
print(f"Baseline data uri: {baseline_data_uri}")
print(f"Baseline results uri: {baseline_results_uri}")

In [ ]:
# Uploading baseline
baseline_dataset_uri = S3Uploader.upload(f"test_data/{validate_dataset}", baseline_data_uri)
baseline_dataset_uri

## Create Baseline Job

In [ ]:
# Setting up model monitor object 
churn_model_quality_monitor = ModelQualityMonitor(
    role=role,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=session,)

In [ ]:
# Naming the baseline job
baseline_job_name = f"xgb-internet-churn-model-baseline-job-{datetime.utcnow():%Y-%m-%d-%H%M}"

In [ ]:
# Execute job, specifying binary classification and prediction/probability columns
job = churn_model_quality_monitor.suggest_baseline(
    job_name=baseline_job_name,
    baseline_dataset=baseline_dataset_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=baseline_results_uri,
    problem_type="BinaryClassification",
    inference_attribute="prediction",
    probability_attribute="probability",
    ground_truth_attribute="label",
)
job.wait(logs=False)

## Viewing Baseline Job

In [ ]:
# Naming baseline ob
baseline_job = churn_model_quality_monitor.latest_baselining_job

In [ ]:
# Viewing metrics
binary_metrics = baseline_job.baseline_statistics().body_dict["binary_classification_metrics"]
pd.json_normalize(binary_metrics).T

In [ ]:
# View contraints
pd.DataFrame(baseline_job.suggested_constraints().body_dict["binary_classification_constraints"]).T

## Continuous Model Monitoring for Model Quality Drift

In [ ]:
# Method to envoke endpoint designed to send data to a specific SageMaker 
# endpoint for model inference
def invoke_endpoint(ep_name, file_name):
    with open(file_name, "r") as f:
        i = 0
        # Read file line by line
        for row in f:
            # Send each row as payload to endpoint
            payload = row.rstrip("\n")
            response = session.sagemaker_runtime_client.invoke_endpoint(
                EndpointName=endpoint_name,
                ContentType="text/csv",
                Body=payload,
                # Have a unique ID for each row
                # Inference ID used to join prediciton with ground truth
                InferenceId=str(i),
            )["Body"].read()
            i += 1
            sleep(1)
# Method to continuously invoke invoke_endpoint function 
def invoke_endpoint_forever():
    while True:
        try:
            invoke_endpoint(endpoint_name, "test_data/test-Copy1.csv")
        # Igore exception and attempt operator again
        except session.sagemaker_runtime_client.exceptions.ValidationError:
            pass
# Thread initialization and start
thread = Thread(target=invoke_endpoint_forever)
thread.start()

In [ ]:
# View captured data
print("Waiting for captures to show up", end="")
for _ in range(120):
    # List files sorted and check if non empty
    # Read last file
    capture_files = sorted(S3Downloader.list(f"{s3_capture_upload_path}/{endpoint_name}"))
    if capture_files:
        capture_file = S3Downloader.read_file(capture_files[-1]).split("\n")
        capture_record = json.loads(capture_file[0])
        # Verify file is a capture file with inference data
        if "inferenceId" in capture_record["eventMetadata"]:
            break
    print(".", end="", flush=True)
    sleep(1)
print()
# Show last three captured files
print("Found Capture Files:")
print("\n ".join(capture_files[-3:]))

In [ ]:
# View contents of a captured file
print("\n".join(capture_file[-3:-1]))

In [ ]:
# Take Python dictionary of captured record and convert to JSON
# formatted string
print(json.dumps(capture_record, indent=2))

## Generating Synthetic Ground Truth

In [ ]:
import random
# Function to generate ground truth data for predictions
def ground_truth_with_id(inference_id):
    # Use this to get consistent results
    random.seed(inference_id)  
    rand = random.random()
    # Return dictionary to represent event data and ground truth data
    return {
        "groundTruthData": {
            # Generate random positive labels for 70% of the time
            "data": "1" if rand < 0.7 else "0",
            "encoding": "CSV",
        },
        "eventMetadata": {
            "eventId": str(inference_id),
        },
        "eventVersion": "0",
    }
# Function to upload ground truth records to S3
def upload_ground_truth(records, upload_time):
    # Serialize each record due to S3 needing a string format
    fake_records = [json.dumps(r) for r in records]
    data_to_upload = "\n".join(fake_records)
    # Constructing S3 URI 
    target_s3_uri = f"{ground_truth_upload_path}/{upload_time:%Y/%m/%d/%H/%M%S}.jsonl"
    print(f"Uploading {len(fake_records)} records to", target_s3_uri)
    # Upload to S3
    S3Uploader.upload_string_as_file_body(data_to_upload, target_s3_uri)

In [ ]:
# Set ground truth records, rows sending in for inference
# Used professor's count
NUM_GROUND_TRUTH_RECORDS = 334  
# Function for continuous generation and uploading process
def generate_fake_ground_truth_forever():
    j = 0
    while True:
        # Generate fake records to generate a consistent ground truth data
        # record for each inference
        fake_records = [ground_truth_with_id(i) for i in range(NUM_GROUND_TRUTH_RECORDS)]
        # Upload fake records to S3
        upload_ground_truth(fake_records, datetime.utcnow())
        # Wait and repeat
        j = (j + 1) % 5
        sleep(60 * 60)  # do this once an hour
# Threading for function to execute in the background
gt_thread = Thread(target=generate_fake_ground_truth_forever)
gt_thread.start()

## Create Monitoring Schedule

In [ ]:
# Creating monitoring schedule 
churn_monitor_schedule_name = (
    f"xgb-internet-churn-monitoring-schedule-{datetime.utcnow():%Y-%m-%d-%H%M}")

In [ ]:
# Set up endpoint input
endpointInput = EndpointInput(
    endpoint_name=predictor.endpoint_name,
    # First element of output
    probability_attribute="0",
    # Threshold for determining cut off for classifying the inference
    probability_threshold_attribute=0.5,
    # S3 destination
    destination="/opt/ml/processing/input_data",)

In [ ]:
# Monitoring schedule that executes every hour
from sagemaker.model_monitor import CronExpressionGenerator
# Get response from monitor 
response = churn_model_quality_monitor.create_monitoring_schedule(
    monitor_schedule_name=churn_monitor_schedule_name,
    endpoint_input=endpointInput,
    output_s3_uri=baseline_results_uri,
    problem_type="BinaryClassification",
    ground_truth_input=ground_truth_upload_path,
    constraints=baseline_job.suggested_constraints(),
    # How frequently the monitoring job should run, enable Cloudwatch as well
    schedule_cron_expression=CronExpressionGenerator.hourly(),
    enable_cloudwatch_metrics=True,)

In [ ]:
# Show schedule
churn_model_quality_monitor.describe_schedule()

In [ ]:
# Note to team: Wait to run until a little bit before top of the hour
executions = churn_model_quality_monitor.list_executions()
executions

In [ ]:
# Wait for first execution
print("Waiting for first execution", end="")
while True:
    execution = churn_model_quality_monitor.describe_schedule().get(
        "LastMonitoringExecutionSummary"
    )
    if execution:
        break
    print(".", end="", flush=True)
    sleep(10)
print()
# Let team know that an execution is found
print("Execution found!")

In [ ]:
while not executions:
    executions = churn_model_quality_monitor.list_executions()
    print(".", end="", flush=True)
    sleep(10)
latest_execution = executions[-1]
# Show most recent execution
latest_execution.describe()

In [ ]:
# Tells team state of execution and status
status = execution["MonitoringExecutionStatus"]

while status in ["Pending", "InProgress"]:
    print("Waiting for execution to finish", end="")
    latest_execution.wait(logs=False)
    latest_job = latest_execution.describe()
    print()
    print(f"{latest_job['ProcessingJobName']} job status:", latest_job["ProcessingJobStatus"])
    print(
        f"{latest_job['ProcessingJobName']} job exit message, if any:",
        latest_job.get("ExitMessage"),
    )
    print(
        f"{latest_job['ProcessingJobName']} job failure reason, if any:",
        latest_job.get("FailureReason"),
    )
    sleep(
        30
    )  
    # Wait for second execution to begin
    latest_execution = churn_model_quality_monitor.list_executions()[-1]
    execution = churn_model_quality_monitor.describe_schedule()["LastMonitoringExecutionSummary"]
    status = execution["MonitoringExecutionStatus"]

print("Execution status is:", status)

if status != "Completed":
    print(execution)
    print(
        "====STOP==== \n No completed executions to inspect further. Please wait till an execution completes or investigate previously reported failures."
    )

In [ ]:
# Show location of the report
latest_execution = churn_model_quality_monitor.list_executions()[-1]
report_uri = latest_execution.describe()["ProcessingOutputConfig"]["Outputs"][0]["S3Output"][
    "S3Uri"
]
print("Report Uri:", report_uri)

In [ ]:
# View violations made by monitoring schedule
pd.options.display.max_colwidth = None
violations = latest_execution.constraint_violations().body_dict["violations"]
violations_df = pd.json_normalize(violations)
violations_df.head(10)

## View Model Quality CloudWatch Metrics

In [ ]:
# Make a CW client 
cw_client = boto3.Session().client("cloudwatch")
namespace = "aws/sagemaker/Endpoints/model-metrics"
cw_dimensions = [
    {"Name": "Endpoint", "Value": endpoint_name},
    {"Name": "MonitoringSchedule", "Value": churn_monitor_schedule_name},]

In [ ]:
# Show CW metrics 
paginator = cw_client.get_paginator("list_metrics")

for response in paginator.paginate(Dimensions=cw_dimensions, Namespace=namespace):
    model_quality_metrics = response["Metrics"]
    for metric in model_quality_metrics:
        print(metric["MetricName"])

In [ ]:
# Create CW alarm for F2 score
alarm_name = "MODEL_QUALITY_F2_SCORE"
alarm_desc = (
    "Trigger an CloudWatch alarm when the f2 score drifts away from the baseline constraints"
)
# Setting low to trigger alarm to see if working
model_quality_f2_drift_threshold = (
    0.625  
metric_name = "f2"
namespace = "aws/sagemaker/Endpoints/model-metrics"

cw_client.put_metric_alarm(
    AlarmName=alarm_name,
    AlarmDescription=alarm_desc,
    ActionsEnabled=True,
    MetricName=metric_name,
    Namespace=namespace,
    Statistic="Average",
    Dimensions=[
        {"Name": "Endpoint", "Value": endpoint_name},
        {"Name": "MonitoringSchedule", "Value": churn_monitor_schedule_name},
    ],
    Period=600,
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=model_quality_f2_drift_threshold,
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="breaching",
)

In [ ]:
# Create CW alarm for recall
alarm_name = "MODEL_QUALITY_RECALL"
alarm_desc = (
    "Trigger a CloudWatch alarm when the recall drifts away from the baseline constraints"
)
# Setting a threshold for recall to trigger an alarm to check if it's working
model_quality_recall_drift_threshold = 0.7  # Example threshold
metric_name = "recall"
namespace = "aws/sagemaker/Endpoints/model-metrics"

cw_client.put_metric_alarm(
    AlarmName=alarm_name,
    AlarmDescription=alarm_desc,
    ActionsEnabled=True,
    MetricName=metric_name,
    Namespace=namespace,
    Statistic="Average",
    Dimensions=[
        {"Name": "Endpoint", "Value": endpoint_name},
        {"Name": "MonitoringSchedule", "Value": churn_monitor_schedule_name},
    ],
    Period=600,  # 10 minutes
    EvaluationPeriods=1,
    DatapointsToAlarm=1,
    Threshold=model_quality_recall_drift_threshold,
    ComparisonOperator="LessThanOrEqualToThreshold",
    TreatMissingData="breaching",
)

In [ ]:
# Delete monitoring scheduleto not incur cost
churn_model_quality_monitor.delete_monitoring_schedule()
sleep(60) 

In [ ]:
!aws sagemaker list-endpoints

In [ ]:
# Delete endpoints to not incur cost
sagemaker = boto3.client('sagemaker')
sagemaker.delete_endpoint(EndpointName="xgb-churn-model-quality-monitor-2024-02-28-0150")